# Pestaña 3 · Optimización de hiperparámetros

In [1]:
import sys
sys.path.insert(0, "dashboard")
import pandas as pd
from IPython.display import HTML, Markdown, display
from src import figuras as fg
from src import textos as tx
from src.datos import num, objeto, pct, tabla

_primera = [True]


def figura(f):
    """Inserta la figura interactiva de Plotly (la librería se carga una vez por página)."""
    display(HTML(f.to_html(full_html=False, include_plotlyjs="cdn" if _primera[0] else False, config=fg.CONFIG,
                           default_width="100%")))
    _primera[0] = False


def texto(parrafos):
    display(Markdown("\n\n".join(parrafos)))


def tab(df, dec=3):
    display(df.style.format(precision=dec, decimal=",", thousands=" ", na_rep="—").hide(axis="index"))

La tercera pestaña compara los métodos de optimización con el mismo presupuesto: la métrica del bucle externo, el costo, la convergencia (curvas *anytime*) y la eficiencia por unidad de presupuesto.

![Vista de la pestaña en el dashboard](../imagenes/pestana_3_optimizacion.png)

Las figuras son interactivas (se puede pasar el cursor sobre ellas) y las interpretaciones usan las mismas cifras que el dashboard y que el cuaderno.

## 3.1 Métrica externa, tiempo y evaluaciones por método

In [2]:
tab(tabla('opt_clf').rename(columns={'index': 'Método'}))
tab(tabla('opt_reg').rename(columns={'index': 'Método'}), dec=4)
texto(tx.optimizadores())

Método,Métrica externa (media ± DE),Evaluaciones por pliegue,Evaluaciones (total),Tiempo total (min),Tiempo por evaluación (s),Pliegues
Grid Search,"0,085 ± 0,065","18,571",2 600,"328,079","7,918",140
Random Search,"0,084 ± 0,064","20,000",2 800,"405,385","8,687",140
Bayesiana (Optuna TPE),"0,084 ± 0,065","20,000",2 800,"390,933","8,369",140
Genético (DEAP),"0,085 ± 0,066","19,857",2 780,"448,661","9,644",140


Método,Métrica externa (media ± DE),Evaluaciones por pliegue,Evaluaciones (total),Tiempo total (min),Tiempo por evaluación (s),Pliegues
Grid Search,"0,916 ± 0,034","18,5714",650,"50,3962","4,9117",35
Random Search,"0,917 ± 0,032","20,0000",700,"57,6188","4,9386",35
Bayesiana (Optuna TPE),"0,917 ± 0,034","20,0000",700,"53,9244","4,6141",35
Genético (DEAP),"0,915 ± 0,033","20,0000",700,"59,0997","5,0648",35


**Interpretación.** Con 20 configuraciones por pliegue, Friedman sobre 140 bloques de clasificación da χ² = 9,34 (p = 0,025); el mejor rango promedio es el de Grid Search (2,23). En regresión, χ² = 0,91 (p = 0,822).

Esto significa que el método de búsqueda cambia de forma sistemática el modelo que llega al bucle externo, aunque mucho menos que la elección del modelo.

Por lo anterior, conviene juzgar la eficiencia por la rapidez con que cada método llega a buenas soluciones (curvas anytime), no solo por el valor final.

## 3.2 Curvas de desempeño en cualquier momento

In [3]:
for g_, e_ in (('clf', 'evaluaciones'), ('clf', 'tiempo'), ('reg', 'evaluaciones'), ('goat', 'evaluaciones')):
    figura(fg.anytime(g_, e_))

In [4]:
figura(fg.optuna_ejemplo())
figura(fg.params_optuna('Random Search'))
figura(fg.params_optuna('Bayesiana (Optuna TPE)'))

## 3.3 Eficiencia cuantitativa

In [5]:
tab(tabla('eficiencia').rename(columns={'optimizador': 'Método', 'tarea': 'Tarea'}))
tab(tabla('eficiencia_pruebas'), dec=4)
texto(tx.eficiencia())

Método,AUAC evaluaciones,AUAC tiempo,Final normalizado,Tarea
Bayesiana (Optuna TPE),"0,763","0,767","0,906",clf
Genético (DEAP),"0,772","0,759","0,883",clf
Grid Search,"0,661","0,697","0,836",clf
Random Search,"0,745","0,733","0,819",clf
Bayesiana (Optuna TPE),"0,950","0,932","0,993",reg
Genético (DEAP),"0,961","0,946","0,988",reg
Grid Search,"0,846","0,834","0,992",reg
Random Search,"0,957","0,938","0,986",reg


Comparación,AUAC frente a,Diferencia media,p crudo,delta de Cliff,p Holm,Magnitud
Bayesiana (Optuna TPE) vs Grid Search,evaluaciones,"0,1015","0,0000","0,3184","0,0000",pequeño
Bayesiana (Optuna TPE) vs Grid Search,tiempo,"0,0701","0,0000","0,1881","0,0000",pequeño
Bayesiana (Optuna TPE) vs Random Search,evaluaciones,"0,0178","0,1315","0,0409","0,2631",despreciable
Bayesiana (Optuna TPE) vs Random Search,tiempo,"0,0346","0,0121","0,1246","0,0362",despreciable
Genético (DEAP) vs Grid Search,evaluaciones,"0,1111","0,0000","0,3369","0,0000",mediano
Genético (DEAP) vs Grid Search,tiempo,"0,0618","0,0081","0,1655","0,0324",pequeño
Genético (DEAP) vs Random Search,evaluaciones,"0,0274","0,0000","0,1116","0,0000",despreciable
Genético (DEAP) vs Random Search,tiempo,"0,0262","0,0004","0,1149","0,0018",despreciable
Bayesiana (Optuna TPE) vs Genético (DEAP),evaluaciones,"-0,0096","0,6031","-0,0994","0,6031",despreciable
Bayesiana (Optuna TPE) vs Genético (DEAP),tiempo,"0,0083","0,6698","-0,0232","0,6698",despreciable


**Interpretación.** El área bajo la curva anytime (1 = llegar al mejor valor desde la primera evaluación) es 0,763 con Bayesiana (Optuna TPE), 0,772 con Genético (DEAP), 0,661 con Grid Search y 0,745 con Random Search; 4 de 6 comparaciones por pares son significativas tras el ajuste de Holm.

Hay que tener en cuenta que los métodos adaptativos superan con claridad a la rejilla, pero frente a Random Search la ventaja es pequeña: las primeras 5 evaluaciones de Optuna y las 6 de la población inicial del genético son aleatorias, así que con 20 evaluaciones su ventaja adaptativa tiene poco margen para aparecer.

Así las cosas, se decide leer la eficiencia como un resultado condicionado al presupuesto y a la dimensión del espacio, con su tamaño de efecto (delta de Cliff) junto al valor p.

## 3.4 Genético, GOAT iterativo, multi-fidelidad y semillas

In [6]:
figura(fg.diversidad_genetico())
tab(tabla('goat_h12'), dec=4)
texto(tx.goat())
tab(tabla('halving'), dec=2)
tab(tabla('semillas_optimizador').rename(columns={'index': 'Método'}), dec=4)

Tarea,Alcance,Bloques,GOAT (media),Optuna (media),Diferencia GOAT − Optuna,p Wilcoxon,delta de Cliff (a favor de GOAT),Evaluaciones GOAT / Optuna,AUAC GOAT / Optuna
Clasificación (AUC-PR),Todos los modelos GOAT,100,"0,0827","0,0813","0,0013","0,2670","0,0055","14,8 / 20,0","0,736 / 0,736"
Clasificación (AUC-PR),Solo SVM / SVR (H2),20,"0,0802","0,0799","0,0003","0,7285","0,0100","16,4 / 20,0","0,820 / 0,820"
Regresión (RMSE),Todos los modelos GOAT,25,"0,9139","0,9141","-0,0002","0,9368","0,0080","15,4 / 20,0","0,961 / 0,966"
Regresión (RMSE),Solo SVM / SVR (H2),5,"0,9180","0,9185","-0,0005","0,8125","0,0400","18,4 / 20,0","0,964 / 0,958"


**Interpretación.** En clasificación, GOAT iterativo logra un AUC-PR externo de 0,0827 frente a 0,0813 de Optuna (p = 0,267), con 14,8 / 20,0 evaluaciones. En la SVM la diferencia es 0,0003 (p = 0,729).

Es decir, GOAT ahorra evaluaciones porque se detiene cuando la mejora ya no supera el error estándar entre pliegues; el arreglo ortogonal estima efectos principales y puede perder interacciones como C × gamma.

Por esta razón, lo indicado es presentarlo como una extensión frugal y exploratoria (el método publicado es de un solo pase), no como un optimizador superior.

Modelo,Método,AUC-PR externo,AUC externo,Configuraciones evaluadas por pliegue,Evaluaciones equivalentes por pliegue,Tiempo por pliegue (s)
Random Forest,Successive Halving,"0,092 ± 0,086","0,534 ± 0,086","39,00","8,91","149,27"
Random Forest,Random Search,"0,088 ± 0,078","0,523 ± 0,081","20,00","20,00","207,35"
Random Forest,Bayesiana (Optuna TPE),"0,095 ± 0,092","0,538 ± 0,079","20,00","20,00","282,27"
XGBoost,Successive Halving,"0,087 ± 0,073","0,528 ± 0,056","39,00","8,91","140,64"
XGBoost,Random Search,"0,088 ± 0,076","0,537 ± 0,065","20,00","20,00","163,32"
XGBoost,Bayesiana (Optuna TPE),"0,088 ± 0,078","0,529 ± 0,066","20,00","20,00","159,39"


Método,AUC-PR externo por semilla (media de 5 pliegues),Desv. est. entre semillas,Desv. est. entre pliegues (semilla 42),Desv. est. de log10(learning_rate) entre semillas (media por pliegue),Desv. est. del puntaje interno entre semillas
Bayesiana (Optuna TPE),"0,0871; 0,0876; 0,0844; 0,0880","0,0017","0,0782","0,3458","0,0021"
Genético (DEAP),"0,0887; 0,0881; 0,0909; 0,0893","0,0012","0,0756","0,3027","0,0015"
Random Search,"0,0875; 0,0925; 0,0826; 0,0885","0,0041","0,0759","0,3693","0,0015"
